In [1]:
from pygom.model.ode_variable import State, Parameter, CallableParameter
import pytest
import numpy as np

# Test Variable Objects

## 1. CallableParameter 

In [2]:
def test_dict_or_tuple(dist, params):
    """
    Check if callable param can be defined via dict or tuple
    """
    x = CallableParameter(
        (dist, params)
    )

    y = CallableParameter(
        (dist, tuple(params.values()))
    )

def test_same_seed(x):
    """
    Test that same seed results in same random draws
    for random process, x
    """
    rng1 = np.random.default_rng(123)
    rng2 = np.random.default_rng(123)

    assert x(rng1) == x(rng2)

def test_different_seed(x):
    """
    Test that different seed results in different random draws
    for random process, x (not guaranteed to always be true)
    """
    rng1 = np.random.default_rng(123)
    rng2 = np.random.default_rng(456)

    assert x(rng1) != x(rng2)

def test_continuous_output(x, rng):
    """
    Test that continuous output is finite and scalar
    """
    sample = x(rng)

    assert np.isscalar(sample)
    assert np.isfinite(sample)

def test_discrete_output(x, rng):
    """
    Test that discrete output is finite and integer
    """
    sample = x(rng)

    assert isinstance(sample, (int, np.integer))

### 1.1.1 Test a few numpy distributions

In [3]:
# Continuous
numpy_test_dists_continuous = {
    'gamma': {'shape': 2.0, 'scale':3.0},
    'normal': {'loc': 0.0, 'scale': 1.0},
    'exponential': {'scale': 10.0},
    'uniform': {'low': 0.0, 'high': 10.0},
    'beta': {'a': 1.0, 'b': 1.0}
}

# Discrete
numpy_test_dists_discrete = {
    'poisson': {'lam': 3.0},
    'binomial': {'n': 10, 'p': 0.5},
    'negative_binomial': {'n': 10, 'p': 0.5}
}

for dist, params in numpy_test_dists_continuous.items():
    test_dict_or_tuple(dist, params)

    x = CallableParameter(
        (dist, params)
    )

    test_same_seed(x)
    test_different_seed(x)
    test_continuous_output(x, np.random.default_rng(1234))

for dist, params in numpy_test_dists_discrete.items():
    test_dict_or_tuple(dist, params)

    x = CallableParameter(
        (dist, params)
    )

    test_same_seed(x)
    test_discrete_output(x, np.random.default_rng(1234))

### 1.1.1 Test a custom distribution

In [4]:
def custom_sampler(rng, lo, hi):
    i1 = rng.uniform(low=lo, high=hi)
    i2 = rng.uniform(low=lo, high=hi)

    return i1 + i2

dist = custom_sampler
params = {'lo': 5, 'hi': 10}

test_dict_or_tuple(dist, params)

x = CallableParameter(
    (dist, params)
)

test_same_seed(x)
test_different_seed(x)
test_continuous_output(x, np.random.default_rng(1234))

## 2. State

In [5]:
x = State(ID='x')
x.initial_value = 5

with pytest.raises(ValueError):
    x.initial_value = -1

# 3. Parameter

In [6]:
x = Parameter(ID='alpha', value=CallableParameter( ('poisson', {'lam': 3.0}) ))

rng = np.random.default_rng(123)

print(x.value)

for _ in range(10):
    x.realise(rng)
    print(x.value)

None
1
1
3
5
5
1
3
6
5
1


# 4. Stores

In [7]:
from pygom.model.ode_utils.variable_store import ParameterStore

parameter_list = ['alpha', 'beta', 'gamma']

parameter_store = ParameterStore()
parameter_store.add(parameter_list, {})

parameter_store.values = {
    'alpha': 1,
    'beta': 1
}

print(parameter_store.values)
parameter_store.all_values_set

[1, 1, None]


True

In [14]:
from pygom.model.ode_utils.variable_store import ParameterStore

parameter_list = ['alpha', 'beta', 'gamma']

rng = np.random.default_rng(123)

parameter_store = ParameterStore(rng=rng)
parameter_store.add(parameter_list, {})

parameter_store.values = {
    'alpha': 3.14,
    'beta': ('exponential', {'scale': 10.0}),
    'gamma': ('exponential', {'scale': 10.0})
}

In [15]:
print(parameter_store.values)
print(parameter_store.values)
parameter_store.new_realisation()
print(parameter_store.values)
print(parameter_store.values)
parameter_store.new_realisation()
print(parameter_store.values)
print(parameter_store.values)

[3.14, 5.969724951226518, 1.1703106104833545]
[3.14, 5.969724951226518, 1.1703106104833545]
[3.14, 2.517946994932384, 3.079654454671414]
[3.14, 2.517946994932384, 3.079654454671414]
[3.14, 0.9180266026312217, 23.516208555408898]
[3.14, 0.9180266026312217, 23.516208555408898]


In [10]:

# def test_default_symbol_created():
#     var = ODEVariable("x")
#     assert str(var.symbol) == "x"

# # def test_symbol_must_be_sympy_symbol():
# #     with pytest.raises(InputError):
# #         ODEVariable("x", symbol="x")

# def test_string_equality():
#     assert ODEVariable("x") == "x"

# def test_symbol_equality():
#     assert ODEVariable("x") == sympy.Symbol("x")

# def test_different_ids_not_equal():
#     assert ODEVariable("x") != ODEVariable("y")

# def test_repr():
#     repr(ODEVariable("x"))